> 在线讲义：[第 32 章 · 数据、张量、序列并行：复制模型、切矩阵、切激活](https://wikieden.github.io/ai-infra-from-constraints/lessons/32-dp-tp-sp.html) · 本书《从约束推导 AI Infra》

# 第 32 章 · 数据、张量、序列并行：复制模型、切矩阵、切激活

配套讲义：仓库里 `ai-infra/notes/lessons/32-dp-tp-sp.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。



Qwen3-32B 的 128K 会话容量、TP 的列 / 行切分、式 (6-4)、二维网格的收集量取自 AII 第 6 章 §6.2.1–§6.2.3。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
KiB, MiB, GiB = 2**10, 2**20, 2**30
HBM = 3350e9
WORK = 2 * GiB / 1e9

**函数 `matvec`**：小工具。

In [ ]:
def matvec(x: list[float], W: list[list[float]]) -> list[float]:
    return [sum(x[i] * W[i][j] for i in range(len(x))) for j in range(len(W[0]))]

**函数 `col_split`**：按列切：每卡用完整输入，各得不同输出元素，拼接。

In [ ]:
def col_split(x, W):
    """按列切：每卡用完整输入，各得不同输出元素，拼接。"""
    return [matvec(x, [[row[j]] for row in W])[0] for j in range(len(W[0]))]

**函数 `row_split`**：按行切：每卡用一个输入特征，各得一份部分和，相加。

In [ ]:
def row_split(x, W):
    """按行切：每卡用一个输入特征，各得一份部分和，相加。"""
    parts = [[x[i] * w for w in W[i]] for i in range(len(x))]
    return parts, [sum(p[j] for p in parts) for j in range(len(W[0]))]

**函数 `sessions_fit`**：TP p 卡时每卡能放下的 128K 会话数（p ≤ 8，KV 按头均分）。

In [ ]:
def sessions_fit(p: int, weights_gb: float = 65.52, sess_gb: float = 34.36, cap: float = 80) -> int:
    """TP p 卡时每卡能放下的 128K 会话数（p ≤ 8，KV 按头均分）。"""
    return int((cap - WORK - weights_gb / p) // (sess_gb / p))

**函数 `step_bytes`**：式 (6-4)：Qwen3-32B 一步的 HBM 访问量。

In [ ]:
def step_bytes(s: int) -> float:
    """式 (6-4)：Qwen3-32B 一步的 HBM 访问量。"""
    return 64 * 930 * MiB + 256 * KiB * (s + 1) + 151936 * 5120 * 2

**函数 `grid_send_mib`**：r×c 网格：横向收集同行输入、纵向收集同列权重，每卡发送 (MiB, MiB)。

In [ ]:
def grid_send_mib(rows: int, cols: int, inp: float = 4, wt: float = 6) -> tuple[float, float]:
    """r×c 网格：横向收集同行输入、纵向收集同列权重，每卡发送 (MiB, MiB)。"""
    return (cols - 1) * inp, (rows - 1) * wt

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：TP 的两种切法，x = [2, 3]，W = [[1, 4], [2, 5]]")
x, W = [2, 3], [[1, 4], [2, 5]]
parts, total = row_split(x, W)
print(f"  按列：{col_split(x, W)}（拼接）；按行：部分和 {parts} → {total}（相加）")

In [ ]:
print("\n第二步：Qwen3-32B 一个 128K 会话 34.36 GB")
print(f"  每 token KV {2 * 64 * 8 * 128 * 2 // KiB} KiB；单卡需要 {65.52 + 34.36 + WORK:.2f} GB")
for p in (2, 4, 8):
    print(f"  TP{p}：每卡 {(65.52 + 34.36) / p + WORK:.2f} GB，最多 {sessions_fit(p)} 个会话")
f = (2 * 5120 * 25600 + 25600 * 5120) * 2
print(f"  FFN 三矩阵 {f // MiB} MiB → TP4 每卡 {f // 4 / MiB:.1f} MiB，{f / HBM * 1e6:.1f} → {f / 4 / HBM * 1e6:.1f} μs")
v = step_bytes(131064)
print(f"  式 (6-4)：一步 {v / 1e9:.2f} GB，{v / HBM * 1e3:.2f} ms")

In [ ]:
print("\n第三步：SP 与二维网格")
print(f"  [8192, 4096] 激活 {8192 * 4096 * 2 // MiB} MiB：四卡各存完整 {4 * 64} MiB，按位置分片共 64 MiB")
for r, c in ((4, 4), (2, 8), (8, 2)):
    a, b = grid_send_mib(r, c)
    print(f"  {r}×{c} 网格：输入 {a:.0f} MiB、权重 {b:.0f} MiB，较大的 {max(a, b):.0f}")

## 3. 代码里才看得到的

1. `sessions_fit` 只对 $p \le 8$ 成立：超过 8 卡 KV 不再按 $1/p$ 下降（KV 头只有 8 个）。
2. 式 (6-4) 的 $s + 1$：这一步要写入一个新 token 的 KV，再读整份。
3. 输出投影 151936 × 5120 × 2 ≈ 1.56 GB，单独一项，比一层还大。

## 4. 自己改一改

- 给 `sessions_fit` 加 $p > 8$ 的 KV 复制规则，算 TP16 每卡能放几个。
- 把 `step_bytes` 改成 Qwen3-30B-A3B（48 层、4 个 KV 头），比较同样 128K 的读取。
- 让 `grid_send_mib` 接受 32 张卡，找最快的网格形状。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 分店各接各的客，训练才要汇意见；
> 分科目并排抄，分单据小计相加；
> 抬钢琴人少抬不动，人多喊口令；KV 头分完再加人就是复制；
> 质检各检各的，要整箱时再收齐。